# 7. Gün: Veri Ön İşleme – II (Aykırı Değer ve Proje Atölyesi)

In [18]:
import pandas as pd 
normal_maas = pd.Series([25000, 27000, 28000, 30000, 32000]) 
hatali_maas = pd.Series([25000, 27000, 28000, 30000, 320000]) 
# Bir sıfır fazla! 
print("Normal Ortalama:", normal_maas.mean(), "| Medyan:", normal_maas.median()) 
print("Hatalı Ortalama:", hatali_maas.mean(), "| Medyan:", hatali_maas.median())

Normal Ortalama: 28400.0 | Medyan: 28000.0
Hatalı Ortalama: 86000.0 | Medyan: 28000.0


In [19]:
# 1. Yöntem: describe() ile min ve max sınırlarını kontrol edin 
normal_maas.describe() 
# 2. Yöntem: En yüksek ve en düşük5 kaydı sıralayarak inceleyin 
normal_maas.sort_values(by="Yas", ascending=False).head(5) 
# En büyük 5 değer 
normal_maas.sort_values(by="Yas", ascending=True).head(5) 
# En küçük 5 değer

TypeError: Series.sort_values() got an unexpected keyword argument 'by'

In [2]:
import pandas as pd 
calisan = pd.DataFrame({ "Ad": ["Ali", "Ayse", "Mehmet", "Zeynep", "Can", "Elif", "Burak"], "Yas": [24, 29, 195, 31, 27, -5, 34], "Maas": [26000, 31000, 29000, 33000, 28000, 30000, 450000] })

## Atolye 1 Çözümü

In [14]:
import pandas as pd

# Örnek çalışan tablosu
calisan = pd.DataFrame({
    "Ad": ["Ali", "Ayse", "Mehmet", "Zeynep", "Can", "Elif", "Burak"],
    "Yas": [24, 29, 195, 31, 27, -5, 34],
    "Maas": [26000, 31000, 29000, 33000, 28000, 30000, 450000]
})

In [13]:
# 1. Özet istatistikler ve şüpheli min/max değerleri
print("--- 1. Özet İstatistikler (describe) ---")
print(calisan.describe().round(1))

print("\nŞüpheli Min/Max Değerleri:")
print(f"Yas  -> Min: {calisan['Yas'].min()} (negatif yaş olamaz), Max: {calisan['Yas'].max()} (insan ömrü için imkansız)")
print(f"Maas -> Min: {calisan['Maas'].min()}, Max: {calisan['Maas'].max()} (diğer maaşlara göre aşırı yüksek/aykırı)")

--- 1. Özet İstatistikler (describe) ---
         Yas      Maas
count    7.0       7.0
mean    47.9   89571.4
std     66.2  158949.5
min     -5.0   26000.0
25%     25.5   28500.0
50%     29.0   30000.0
75%     32.5   32000.0
max    195.0  450000.0

Şüpheli Min/Max Değerleri:
Yas  -> Min: -5 (negatif yaş olamaz), Max: 195 (insan ömrü için imkansız)
Maas -> Min: 26000, Max: 450000 (diğer maaşlara göre aşırı yüksek/aykırı)


In [5]:
# 2. Maaş ortalaması (mean) ve medyanı (median) karşılaştırması
maas_ortalama = calisan["Maas"].mean()
maas_medyan = calisan["Maas"].median()

print("\n--- 2. Maaş Ortalama vs Medyan ---")
print(f"Maaş Ortalaması (mean)  : {maas_ortalama:,.2f} TL")
print(f"Maaş Medyanı (median)   : {maas_medyan:,.2f} TL")
print(f"Aradaki Fark            : {abs(maas_ortalama - maas_medyan):,.2f} TL")
# Yorum: 450.000 TL'lik aykırı değer ortalamayı ciddi şekilde yukarı çekerken, medyan bu uç değerden etkilenmez.



--- 2. Maaş Ortalama vs Medyan ---
Maaş Ortalaması (mean)  : 89,571.43 TL
Maaş Medyanı (median)   : 30,000.00 TL
Aradaki Fark            : 59,571.43 TL


In [6]:
# 3. Kesin veri girişi hatası olan kayıtları filtreleme
# Yaşın 0'dan küçük veya 120'den büyük olması kesin veri girişi hatasıdır.
kesin_hatalar = calisan[(calisan["Yas"] < 0) | (calisan["Yas"] > 120)]

In [7]:
print("\n--- 3. Kesin Veri Girişi Hatası Olan Kayıtlar ---")
print(kesin_hatalar)
# Yorum: Mehmet (Yas: 195) ve Elif (Yas: -5) kesin veri girişi hatasıdır.
# Burak'ın maaşı (450.000 TL) aykırı bir değerdir ancak üst düzey yönetici maaşı olabileceğinden "kesin hata" yerine "şüpheli/aykırı kayıt" olarak değerlendirilir.


--- 3. Kesin Veri Girişi Hatası Olan Kayıtlar ---
       Ad  Yas   Maas
2  Mehmet  195  29000
5    Elif   -5  30000


## 2. bölüm

Mantıksal Sınırlarla Aykırı Değer Temizliği

In [8]:
# 1. Mantıklı yaş aralığını filtreleme (18 ile 65 yaş arası) 
temiz = calisan[(calisan["Yas"] >= 18) & (calisan["Yas"] <= 65)] 
# 2. Üst maaş sınırı koyarak aşırı uç kaydı ayıklama 
temiz = temiz[temiz["Maas"] <= 100000] 
# 3. Alternatif: Silmek yerine sınır değerine baskılama (clip) 
calisan["Yas"] = calisan["Yas"].clip(lower=18, upper=65)

## atolye 2 çözümü

In [9]:
import pandas as pd

# Atölye 1'deki çalışan tablosu
calisan = pd.DataFrame({
    "Ad": ["Ali", "Ayse", "Mehmet", "Zeynep", "Can", "Elif", "Burak"],
    "Yas": [24, 29, 195, 31, 27, -5, 34],
    "Maas": [26000, 31000, 29000, 33000, 28000, 30000, 450000]
})

In [10]:
# 1. Yaşı 18 ile 65 arasında olan çalışanları filtreleyip 'temiz' tablosuna aktarma
temiz = calisan[(calisan["Yas"] >= 18) & (calisan["Yas"] <= 65)]

In [11]:
# 2. Maaşı 100.000 TL'den küçük olanları süzerek uç maaş kaydını çıkarma
temiz = temiz[temiz["Maas"] < 100000]

In [12]:
# 3. Temizlik öncesi ve sonrası maaş ortalamalarını karşılaştırma
eski_ortalama = calisan["Maas"].mean()
yeni_ortalama = temiz["Maas"].mean()

In [13]:
print("--- 3. Maaş Ortalaması Karşılaştırması ---")
print(f"Temizlik Öncesi Ortalama : {eski_ortalama:,.2f} TL")
print(f"Temizlik Sonrası Ortalama: {yeni_ortalama:,.2f} TL")
print(f"Ortalama Değişimi        : {yeni_ortalama - eski_ortalama:,.2f} TL")

--- 3. Maaş Ortalaması Karşılaştırması ---
Temizlik Öncesi Ortalama : 89,571.43 TL
Temizlik Sonrası Ortalama: 29,500.00 TL
Ortalama Değişimi        : -60,071.43 TL


In [14]:
# 4. Başlangıçtaki 7 satırdan kaç temiz kayıt kaldığını yazdırma
print("\n--- 4. Temiz Tablo Boyutu ve Kayıtlar ---")
print("Temiz tablo boyutu (satır, sütun):", temiz.shape)
print(f"Başlangıçtaki {calisan.shape[0]} satırdan {temiz.shape[0]} temiz kayıt kaldı.")
print("\nTemizlenmiş Tablo:")
print(temiz)


--- 4. Temiz Tablo Boyutu ve Kayıtlar ---
Temiz tablo boyutu (satır, sütun): (4, 3)
Başlangıçtaki 7 satırdan 4 temiz kayıt kaldı.

Temizlenmiş Tablo:
       Ad  Yas   Maas
0     Ali   24  26000
1    Ayse   29  31000
3  Zeynep   31  33000
4     Can   27  28000
